# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [13]:
df = pd.read_csv("./data/AviationData.csv", encoding="latin1", low_memory=False)
df.info()
df.isna()
df.describe(include='all').T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50249 non-null  object 
 9   Airport.Name            52790 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87572 non-null  object 
 14  Make                    88826 non-null

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Event.Id,88889,87951,20001212X19172,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Investigation.Type,88889,2,Accident,85015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Accident.Number,88889,88863,CEN22LA149,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event.Date,88889,14782,1984-06-30,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,88837,27758,"ANCHORAGE, AK",434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,88663,219,United States,82248,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,34382,25589,332739N,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Longitude,34373,27154,0112457W,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Code,50249,10375,NONE,1488,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Name,52790,24871,Private,240,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [14]:
df['Event.Date'] = pd.to_datetime(df['Event.Date'])

df = df[df['Investigation.Type'] == 'Accident']
df = df[df['Amateur.Built'] == 'No']
df = df[df['Event.Date'].dt.year >= 1983]
df = df.dropna(subset=['Make', 'Model'])

df.shape


(73247, 31)

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [15]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

df = df.dropna(subset=injury_cols, how='all')
df[injury_cols] = df[injury_cols].fillna(0)

df['Total.People'] = df[injury_cols].sum(axis=1)
df = df[df['Total.People'] > 0]

df['Fatal.Serious.Fraction'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.People']

df['Fatal.Serious.Fraction'].describe()

count    72819.000000
mean         0.286003
std          0.434093
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000
Name: Fatal.Serious.Fraction, dtype: float64

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [16]:
df['Aircraft.damage'].value_counts(dropna=False)


Substantial    55381
Destroyed      15362
NaN             1350
Minor            647
Unknown           79
Name: Aircraft.damage, dtype: int64

In [17]:
df['Aircraft.damage'] = df['Aircraft.damage'].replace('Unknown', np.nan)
df['Destroyed'] = df['Aircraft.damage'].map({'Destroyed': 1, 'Substantial': 0, 'Minor': 0})

df['Destroyed'].value_counts(dropna=False)

0.0    56028
1.0    15362
NaN     1429
Name: Destroyed, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [18]:
df['Make'].nunique()

2235

In [19]:
df['Make'] = df['Make'].str.strip().str.title()

make_alias = {
    'Airbus Industrie': 'Airbus',
    'Air Tractor Inc': 'Air Tractor',
    'Aviat Aircraft Inc': 'Aviat',
    'Cirrus Design Corp': 'Cirrus',
    'Cirrus Design Corp.': 'Cirrus',
    'Dehavilland': 'De Havilland',
    'Robinson Helicopter': 'Robinson',
    'Robinson Helicopter Company': 'Robinson',
    'Rockwell International': 'Rockwell',
}
df['Make'] = df['Make'].replace(make_alias)

make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)]

print(df['Make'].nunique(), 'makes remain')
df.shape

74 makes remain


(66496, 34)

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [20]:
df['Model'].isna().sum()

0

In [21]:
df['Model'] = df['Model'].str.strip().str.upper()
(df.groupby('Model')['Make'].nunique() > 1).sum()

313

In [22]:
df['Make.Model'] = df['Make'] + ' ' + df['Model']
df['Make.Model'].nunique()

5765

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [23]:
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(f'--- {col} ---')
    print(df[col].value_counts(dropna=False))
    print()

--- Engine.Type ---
Reciprocating    54377
NaN               4006
Turbo Shaft       2893
Turbo Prop        2444
Unknown           1363
Turbo Fan         1093
Turbo Jet          311
None                 6
LR                   1
UNK                  1
NONE                 1
Name: Engine.Type, dtype: int64

--- Weather.Condition ---
VMC    58484
IMC     5016
NaN     2225
UNK      594
Unk      177
Name: Weather.Condition, dtype: int64

--- Number.of.Engines ---
1.0    53941
2.0     7985
NaN     3490
0.0      705
4.0      198
3.0      176
8.0        1
Name: Number.of.Engines, dtype: int64

--- Purpose.of.flight ---
Personal                     36621
Instructional                 9269
Unknown                       4288
Aerial Application            4127
NaN                           3325
Business                      3320
Positioning                   1367
Other Work Use                1040
Aerial Observation             703
Ferry                          619
Public Aircraft                6

In [24]:
df['Engine.Type'] = df['Engine.Type'].replace('UNK', 'Unknown')
df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': 'Unknown', 'Unk': 'Unknown'})
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace('Air Race show', 'Air Race/show')

df[['Engine.Type', 'Weather.Condition', 'Purpose.of.flight']].nunique()

Engine.Type           9
Weather.Condition     3
Purpose.of.flight    25
dtype: int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [25]:
df.isna().mean().sort_values(ascending=False)

Schedule                  0.874398
Air.carrier               0.829945
FAR.Description           0.704975
Aircraft.Category         0.703651
Longitude                 0.625917
Latitude                  0.625842
Airport.Code              0.429169
Airport.Name              0.402355
Broad.phase.of.flight     0.274212
Publication.Date          0.171153
Engine.Type               0.060244
Report.Status             0.056154
Number.of.Engines         0.052484
Purpose.of.flight         0.050003
Weather.Condition         0.033461
Aircraft.damage           0.017565
Destroyed                 0.017565
Registration.Number       0.012332
Country                   0.002647
Location                  0.000496
Total.Minor.Injuries      0.000000
Total.People              0.000000
Fatal.Serious.Fraction    0.000000
Total.Uninjured           0.000000
Event.Id                  0.000000
Total.Serious.Injuries    0.000000
Total.Fatal.Injuries      0.000000
Investigation.Type        0.000000
Amateur.Built       

In [26]:
cols_to_drop = [
    'Schedule', 'Air.carrier', 'FAR.Description', 'Aircraft.Category',
    'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name',
    'Event.Id', 'Accident.Number', 'Registration.Number', 'Report.Status',
]
df = df.drop(columns=cols_to_drop)
df.shape

(66496, 23)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [27]:
df.to_csv('data/cleaned_aviation_data.csv', index=False)